# GRPO Reward Function Design Workshop

Design a reward function for GRPO (Group Relative Policy Optimization) that
teaches an LLM to solve ASCII mazes.

**No GPU required.** This notebook uses pre-generated model rollouts to let
you iterate on reward functions instantly — all computation is pure Python.

## Background

A small LLM (Qwen2.5-0.5B) has been fine-tuned with SFT to understand the
maze format. It solves ~80% of 3×3 mazes and ~30% of 4×4, but struggles
with larger ones. GRPO can improve it — but only if the reward function
provides useful signal.

## How GRPO works

For each maze, GRPO generates **G rollouts** (attempts) and scores them
with your reward function. It then computes **advantages** by comparing
each rollout to the group mean:

- Rollouts scoring above the mean get **positive** advantage → reinforced
- Rollouts scoring below the mean get **negative** advantage → discouraged
- If all rollouts score the same → advantages are zero → **no learning**

Your job: design a reward function that creates meaningful **variance**
between rollouts while being **informative** (better paths score higher).

## Setup

Run this notebook anywhere — Google Colab, Jupyter, VS Code, etc.
The only dependency is `numpy`.

In [ ]:
!pip install -q numpy

import os, sys
if not os.path.exists('ascii-maze-rl'):
    !git clone -q https://github.com/StephenJHardy/ascii-maze-rl.git
sys.path.insert(0, 'ascii-maze-rl')
print('Ready.')

## 1. Explore the Maze Format

Mazes are rendered as text grids. `#` is a wall, `.` is open, `>` marks
entry (left) and exit (right). The model must output space-separated
moves: `u` (up), `d` (down), `l` (left), `r` (right).

In [ ]:
from src.maze_gen import generate
from src.maze_repr import to_str, solution_to_str, SYSTEM_PROMPT
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress

maze = generate(4, 4, seed=42)
print("Maze (4×4):")
print(to_str(maze))
print(f"\nSolution: {solution_to_str(maze.solution_moves)}")
print(f"Solution length: {len(maze.solution_moves)} moves")
print(f"Entry: {maze.entry}, Exit: {maze.exit}")

In [ ]:
# See how move parsing and simulation work
test_output = "r r d d d r"
moves = extract_moves(test_output)
print(f"Parsed moves: {moves}")

path = simulate(moves, maze)
print(f"Path through maze: {path}")
print(f"Reached exit: {reached_exit(path, maze)}")
print(f"Valid steps: {len(path) - 1} / {len(moves)}")
print(f"Manhattan progress: {manhattan_progress(path[-1], maze.exit, maze.entry):.2f}")

In [ ]:
# Try a wrong path — simulation stops at the first wall hit
bad_output = "d d d d r r r"
moves = extract_moves(bad_output)
path = simulate(moves, maze)
print(f"Moves: {moves}")
print(f"Path: {path}")
print(f"Valid steps: {len(path) - 1} / {len(moves)} (hit a wall)")
print(f"Reached exit: {reached_exit(path, maze)}")

## 2. Design Your Reward Function ⭐

**This is the main exercise.** Edit `maze_reward` below, then re-run
the scoring cells to see how it performs.

### Available tools
- `extract_moves(text)` → parse model output to list of moves, or `None` if unparseable
- `simulate(moves, maze)` → walk through maze, returns path (stops at walls)
- `reached_exit(path, maze)` → did the path reach the exit?
- `manhattan_progress(pos, target, origin)` → progress toward exit (0–1)
- `bfs_solve(width, height, walls, start, end)` → BFS shortest path through the maze

In [ ]:
from src.maze_gen import Maze, solve as bfs_solve
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress


def reconstruct_maze(walls, entry, exit_, width, height, solution_path):
    frozen_walls = frozenset(frozenset(tuple(c) for c in w) for w in walls)
    return Maze(width=width, height=height, walls=frozen_walls,
               entry=tuple(entry), exit=tuple(exit_),
               solution=tuple(tuple(p) for p in solution_path), seed=0)


def maze_reward(completions, **kwargs):
    """
    Your reward function. Scores each completion against its maze.
    Start simple (v1), then iterate based on the analysis below.

    Args:
        completions: list of model output strings
        **kwargs: maze metadata (walls, entry, exit, width, height, solution_path)

    Returns:
        list of float rewards
    """
    rewards = []
    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            completion = completion[-1]["content"] if completion else ""
        elif isinstance(completion, dict):
            completion = completion.get("content", "")

        maze = reconstruct_maze(
            walls=kwargs["walls"][i], entry=kwargs["entry"][i],
            exit_=kwargs["exit"][i], width=kwargs["width"][i],
            height=kwargs["height"][i], solution_path=kwargs["solution_path"][i],
        )

        # --- v1: naive binary reward (start here) ---
        moves = extract_moves(completion)
        if moves is None:
            reward = 0.0
        else:
            path = simulate(moves, maze)
            reward = 1.0 if reached_exit(path, maze) else 0.0

        rewards.append(reward)
    return rewards

### 2b. Score Pre-generated Rollouts

These rollouts were generated by the SFT model at multiple temperatures.
Re-run this cell after changing your reward function — scoring is instant.

In [ ]:
import json as _json, numpy as np

with open('ascii-maze-rl/configs/pregenerated_rollouts.json') as f:
    rollout_data = _json.load(f)

mazes_meta = rollout_data["mazes"]
all_completions = rollout_data["completions"]
temperatures = rollout_data["temperatures"]

print(f"{'Temp':>4s} | {'Mean Reward':>11s} | {'Mean Std':>9s} | "
      f"{'Solved':>8s} | {'Zero-var mazes':>14s}")
print("-" * 60)

for temp in temperatures:
    temp_key = str(temp)
    all_stds, all_rewards = [], []
    total_solved, zero_var = 0, 0
    total = len(mazes_meta) * 8

    for maze_idx, meta in enumerate(mazes_meta):
        comps = all_completions[temp_key][maze_idx]
        kwargs = {
            "walls": [meta["walls"]] * 8,
            "entry": [meta["entry"]] * 8,
            "exit": [meta["exit"]] * 8,
            "width": [meta["width"]] * 8,
            "height": [meta["height"]] * 8,
            "solution_path": [meta["solution_path"]] * 8,
        }
        rewards = maze_reward(comps, **kwargs)
        std = np.std(rewards)
        all_stds.append(std)
        all_rewards.append(np.mean(rewards))
        if std < 0.005:
            zero_var += 1
        maze = reconstruct_maze(meta["walls"], meta["entry"], meta["exit"],
                                meta["width"], meta["height"], meta["solution_path"])
        for c in comps:
            moves = extract_moves(c)
            if moves and reached_exit(simulate(moves, maze), maze):
                total_solved += 1

    print(f"{temp:4.1f} | {np.mean(all_rewards):>11.4f} | {np.mean(all_stds):>9.4f} | "
          f"{total_solved:>3d}/{total:<3d} | {zero_var:>6d}/{len(mazes_meta):<6d}")

print("\nGood reward functions have:")
print("   - Mean Std > 0.15 at temp=1.0")
print("   - Few zero-variance mazes")
print("   - If ALL mazes are zero-var, GRPO learns nothing!")

### 2c. Visualize Rollouts

See how your reward scores different rollouts, and which ones GRPO
would reinforce (positive advantage) vs discourage (negative advantage).

In [ ]:
TEMP = "1.0"  # try: "0.6", "0.8", "1.0", "1.2", "1.4", "1.6", "1.8"

from src.rollout_capture import MazeRollouts, RolloutResult, compute_advantages
from src.build_rollout_viewer import HTML_TEMPLATE
from dataclasses import asdict
from IPython.display import HTML, display

viz_rollouts = []
for maze_idx, meta in enumerate(mazes_meta):
    comps = all_completions[TEMP][maze_idx]
    maze = reconstruct_maze(meta["walls"], meta["entry"], meta["exit"],
                            meta["width"], meta["height"], meta["solution_path"])
    kwargs = {
        "walls": [meta["walls"]] * 8, "entry": [meta["entry"]] * 8,
        "exit": [meta["exit"]] * 8, "width": [meta["width"]] * 8,
        "height": [meta["height"]] * 8, "solution_path": [meta["solution_path"]] * 8,
    }
    rewards = maze_reward(comps, **kwargs)
    advantages, mean_r, std_r = compute_advantages(rewards)

    rollouts = []
    for j, c in enumerate(comps):
        moves = extract_moves(c)
        path = simulate(moves or [], maze)
        rollouts.append(RolloutResult(
            completion=c, moves_parsed=moves,
            path=[list(p) for p in path], reward=rewards[j],
            solved=reached_exit(path, maze),
            valid_steps=len(path)-1,
            progress=max(manhattan_progress(path[-1], maze.exit, maze.entry), 0.0),
        ))

    viz_rollouts.append(MazeRollouts(
        maze_id=meta["id"], width=meta["width"], height=meta["height"],
        maze_str=meta["maze_str"], entry=meta["entry"], exit=meta["exit"],
        correct_path=meta["solution_path"],
        correct_moves=meta["solution_moves"].split() if isinstance(meta["solution_moves"], str) else meta["solution_moves"],
        solution_length=meta["solution_length"],
        rollouts=rollouts, advantages=advantages,
        reward_mean=mean_r, reward_std=std_r,
    ))

import json as _json
viewer_data = [asdict(r) for r in viz_rollouts]
serialized = _json.dumps(viewer_data).replace('</', '<\\/')
viewer_html = HTML_TEMPLATE.replace('__DATA_PLACEHOLDER__', serialized)

with open('rollout_viewer.html', 'w') as f:
    f.write(viewer_html)
print(f'Rollout viewer (temp={TEMP}): {len(viz_rollouts)} mazes')
print('Open rollout_viewer.html in a browser for the full interactive viewer.')

display(HTML(f'<iframe srcdoc="{viewer_html.replace(chr(34), "&quot;")}" '
            f'width="100%" height="600px" style="border:1px solid #333;'
            f'border-radius:4px;"></iframe>'))

### 2d. Iterate!

Go back to the reward function cell, improve it, re-run the scoring.

**Progression:**
1. **Binary (0/1)** → notice high zero-var count. GRPO is blind.
2. **Penalize gibberish** (`-1.0` for unparseable) → differentiates bad output
3. **Partial credit** (coverage + progress) → reduces zero-variance groups
4. **BFS distance** (actual maze distance, not manhattan) → ignores walls
5. **Loop penalty** → discourages random wandering that revisits cells

### What to look for in the scoring table
- **Mean Std > 0.15** at temp=1.0 → GRPO has signal to learn from
- **Few zero-var mazes** → most groups have useful contrast
- **Higher temp = more variance** but also more gibberish at 1.4+

## 3. Understanding GRPO Mechanics

Let's look at a concrete example of how GRPO uses your reward function.

In [ ]:
# Pick a maze and show how advantages are computed
example_idx = 5  # a 4x4 maze
meta = mazes_meta[example_idx]
comps = all_completions["1.0"][example_idx]
maze = reconstruct_maze(meta["walls"], meta["entry"], meta["exit"],
                        meta["width"], meta["height"], meta["solution_path"])

print(f"Maze {meta['id']} ({meta['width']}×{meta['height']}):")
print(meta["maze_str"])
print(f"\nCorrect solution: {meta['solution_moves']} ({meta['solution_length']} moves)")

# Score with your reward function
kwargs = {
    "walls": [meta["walls"]] * 8, "entry": [meta["entry"]] * 8,
    "exit": [meta["exit"]] * 8, "width": [meta["width"]] * 8,
    "height": [meta["height"]] * 8, "solution_path": [meta["solution_path"]] * 8,
}
rewards = maze_reward(comps, **kwargs)
advantages, mean_r, std_r = compute_advantages(rewards)

print(f"\n{'Rollout':>7s}  {'Output':>25s}  {'Reward':>7s}  {'Advantage':>10s}  {'GRPO signal':>12s}")
print("-" * 75)
for j, (c, r, a) in enumerate(zip(comps, rewards, advantages)):
    signal = "REINFORCE" if a > 0.01 else "DISCOURAGE" if a < -0.01 else "(ignored)"
    print(f"     {j+1}  {c[:25]:>25s}  {r:7.3f}  {a:+10.3f}  {signal:>12s}")

print(f"\nGroup mean: {mean_r:.3f}, std: {std_r:.3f}")
if std_r < 0.01:
    print("⚠️  Zero variance — GRPO learns NOTHING from this group!")
else:
    print("✓ Variance exists — GRPO can differentiate good from bad rollouts.")

## Solution (if stuck)

The best reward function uses BFS distance (actual maze distance to exit),
not manhattan distance. It also penalizes loops and gibberish.

In [ ]:
# =================================================================
# SOLUTION: BFS-distance reward function
# =================================================================
# Uncomment and use this if you get stuck. But try designing your
# own first — the learning is in the iteration!
#
# def maze_reward(completions, **kwargs):
#     from src.maze_gen import solve as bfs_solve
#     rewards = []
#     for i, completion in enumerate(completions):
#         if isinstance(completion, list):
#             completion = completion[-1]["content"] if completion else ""
#         elif isinstance(completion, dict):
#             completion = completion.get("content", "")
#
#         maze = reconstruct_maze(
#             walls=kwargs["walls"][i], entry=kwargs["entry"][i],
#             exit_=kwargs["exit"][i], width=kwargs["width"][i],
#             height=kwargs["height"][i], solution_path=kwargs["solution_path"][i],
#         )
#
#         moves = extract_moves(completion)
#         if moves is None:
#             rewards.append(-1.0)
#             continue
#
#         path = simulate(moves, maze)
#         valid_steps = len(path) - 1
#         optimal_len = len(maze.solution) - 1
#
#         if reached_exit(path, maze):
#             efficiency = optimal_len / max(len(moves), optimal_len)
#             rewards.append(0.6 + 0.4 * efficiency)
#             continue
#
#         final_pos = path[-1]
#         remaining = bfs_solve(maze.width, maze.height, maze.walls, final_pos, maze.exit)
#         remaining_dist = len(remaining) - 1 if remaining else optimal_len
#         bfs_progress = max(1.0 - remaining_dist / max(optimal_len, 1), 0.0)
#
#         unique_cells = len(set(path))
#         loop_penalty = 1.0 - (len(path) - unique_cells) / max(len(path), 1)
#
#         coverage = min(valid_steps / max(optimal_len, 1), 1.0)
#         partial = 0.5 * (0.7 * bfs_progress + 0.2 * loop_penalty + 0.1 * coverage)
#         rewards.append(partial)
#
#     return rewards
#
# Key design principles:
#   - BFS distance: measures actual maze distance, not manhattan (ignores walls)
#   - Loop penalty: discourages random wandering that revisits cells
#   - Smooth gradient: every step closer to exit increases reward
#   - Clear gap: solved (0.6+) vs partial (0-0.5) vs gibberish (-1.0)

## Key Takeaways

1. **GRPO uses group comparison** — it doesn't need a value model, just
   relative ranking within rollout groups
2. **Reward design is critical** — binary rewards give sparse signal;
   partial credit enables smooth optimization
3. **Variance is everything** — if all rollouts score the same, GRPO is
   blind. Your reward must create contrast between good and bad attempts.
4. **Credit assignment is per-sequence** — GRPO can't tell which move
   was wrong, only whether the whole attempt was better or worse
   (this motivates value models / process reward models)